# Part 1

# 1. Import libraries


In [1]:
import pandas as pd
import numpy as np
import re
import statsmodels.api as sm


# 2. Load the data

In [2]:
calls_df = pd.read_csv(
    "https://www.dropbox.com/scl/fi/2p7ahxroqj9pwf98ni5an/Sample_Calls.csv?rlkey=zfieicvz891u4e3z0aroeg0u7&dl=1",
    low_memory=False
)

presentation_df = pd.read_feather(
    "https://www.dropbox.com/scl/fi/uceh2xva5g4apbmt92cgt/Sample_Calls_Presentations.feather?rlkey=ln4nzsa4nenqyvm0pg2cur9sp&dl=1"
)

qa_df = pd.read_feather(
    "https://www.dropbox.com/scl/fi/iq4111nlmsykp2tzxk9xg/Sample_Calls_QA.feather?rlkey=xabjqmwhesx05jivrlfzkgj6m&dl=1"
)

calls_df["date_rdq"] = pd.to_datetime(calls_df["date_rdq"], errors="coerce")

print("calls_df shape:", calls_df.shape)
print("presentation_df shape:", presentation_df.shape)
print("qa_df shape:", qa_df.shape)


calls_df shape: (2877, 45)
presentation_df shape: (2877, 2)
qa_df shape: (163769, 5)


# 3. Build one main dataset

Creates one row per call and adds:


1.   presentation text
2.   all Q&A text
3.   question-only text
4.   answer-only text
5.   full call text = presentation + all Q&A


In [3]:
# Rename the presentation column so the meaning is clear
presentation_df = presentation_df.rename(columns={"presentation": "presentation_text"})

# Clean QA type labels
qa_df["QA"] = qa_df["QA"].fillna("").astype(str).str.lower().str.strip()
qa_df["QA_text"] = qa_df["QA_text"].fillna("").astype(str)

# Keep QA rows in order
qa_df = qa_df.sort_values(["file_name", "QA_number", "QA"])

# Combine all Q&A text for each call
qa_all_text = qa_df.groupby("file_name", as_index=False)["QA_text"].apply(" ".join)
qa_all_text = qa_all_text.rename(columns={"QA_text": "qa_text_all"})

# Combine only question text
qa_questions_text = qa_df[qa_df["QA"] == "q"].groupby("file_name", as_index=False)["QA_text"].apply(" ".join)
qa_questions_text = qa_questions_text.rename(columns={"QA_text": "qa_questions_text"})

# Combine only answer text
qa_answers_text = qa_df[qa_df["QA"] == "a"].groupby("file_name", as_index=False)["QA_text"].apply(" ".join)
qa_answers_text = qa_answers_text.rename(columns={"QA_text": "qa_answers_text"})

# Merge everything into one main dataframe
master_df = calls_df.merge(presentation_df, on="file_name", how="left")
master_df = master_df.merge(qa_all_text, on="file_name", how="left")
master_df = master_df.merge(qa_questions_text, on="file_name", how="left")
master_df = master_df.merge(qa_answers_text, on="file_name", how="left")

# Fill missing text with empty strings
text_cols = [
    "presentation_text",
    "qa_text_all",
    "qa_questions_text",
    "qa_answers_text"
]

for col in text_cols:
    master_df[col] = master_df[col].fillna("").astype(str)

# Create full-call text
master_df["full_call_text"] = (
    master_df["presentation_text"].str.strip() + " " + master_df["qa_text_all"].str.strip()
)
master_df["full_call_text"] = master_df["full_call_text"].str.replace(r"\s+", " ", regex=True).str.strip()

print("master_df shape:", master_df.shape)
print("unique calls:", master_df["file_name"].nunique())
print(master_df[["file_name", "presentation_text", "qa_answers_text", "full_call_text"]].head())

master_df shape: (2877, 50)
unique calls: 2877
                                           file_name  \
0  Download ECC/SE/TRANSCRIPT/XMLStd/Archive/2016...   
1  Download ECC/SE/TRANSCRIPT/XMLStd/Archive/2016...   
2  Download ECC/SE/TRANSCRIPT/XMLStd/Archive/2016...   
3  Download ECC/SE/TRANSCRIPT/XMLStd/Archive/2017...   
4  Download ECC/SE/TRANSCRIPT/XMLStd/Archive/2017...   

                                   presentation_text  \
0      Good morning and thank you for standing by...   
1      Good morning and thank you for standing by...   
2      Good morning and thank you for standing by...   
3      Good morning and thank you for standing by...   
4      Good morning, and thank you for standing b...   

                                     qa_answers_text  \
0  Jeff, this is Rick. I will cover the first and...   
1  Okay. Hi, Jeff, it's Rick. So I'll take I gues...   
2  Sure, Jami; this is Rick. Thank you for the qu...   
3  Hi, Jami; it's Bill. So on your operating marg...   

# 4. Clean the text

Turns text into a standard format:

*  lowercase
*  remove punctuation
*  remove extra spaces
*  count words


In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[\r\n]+", " ", text)
    text = re.sub(r"[^a-z0-9'\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def count_words(text):
    if text == "":
        return 0
    return len(text.split())

main_text_cols = ["presentation_text", "qa_answers_text", "full_call_text"]

for col in main_text_cols:
    clean_col = col.replace("_text", "_clean")
    word_count_col = col.replace("_text", "_n_words")

    master_df[clean_col] = master_df[col].apply(clean_text)
    master_df[word_count_col] = master_df[clean_col].apply(count_words)

print(master_df[[
    "presentation_n_words",
    "qa_answers_n_words",
    "full_call_n_words"
]].describe())


       presentation_n_words  qa_answers_n_words  full_call_n_words
count           2877.000000         2877.000000        2877.000000
mean            3353.573166         4077.648245        8950.563087
std             1313.159181         1501.286157        2161.426155
min                0.000000          158.000000        2442.000000
25%             2457.000000         3102.000000        7718.000000
50%             3245.000000         3879.000000        8787.000000
75%             4136.000000         4837.000000        9803.000000
max            13953.000000        14161.000000       22780.000000


# 5. Load Harvard and LM dictionaries

Loads two sentiment dictionaries:
*   Harvard GI
*   Loughran-McDonald




In [5]:
# ---------- Harvard ----------
harvard_df = pd.read_csv(
    "https://www.dropbox.com/s/wjucnpw39uuxupf/HarvardGI4.txt?dl=1",
    delimiter="\t",
    low_memory=False
)

def normalize_harvard_word(word):
    if pd.isna(word):
        return None

    word = str(word).strip().lower()
    word = word.split("#")[0]   # remove sense marker like able#1
    word = re.sub(r"[^a-z0-9']", "", word)

    if word == "":
        return None
    return word

def harvard_flag_is_active(value):
    if pd.isna(value):
        return False
    value = str(value).strip().lower()
    return value not in ["", "nan", "none", "0"]

harvard_df["word_norm"] = harvard_df["Entry"].apply(normalize_harvard_word)

harvard_pos = set(
    harvard_df.loc[harvard_df["Positiv"].apply(harvard_flag_is_active), "word_norm"].dropna()
)

harvard_neg = set(
    harvard_df.loc[harvard_df["Negativ"].apply(harvard_flag_is_active), "word_norm"].dropna()
)

print("Harvard positive words:", len(harvard_pos))
print("Harvard negative words:", len(harvard_neg))


# ---------- Loughran-McDonald ----------
lm_df = pd.read_csv(
    "https://drive.google.com/uc?id=1ptUgGVeeUGhCbaKL14Ri3Xi5xOKkPkUD",
    low_memory=False
)

lm_df["Word"] = lm_df["Word"].astype(str).str.lower().str.strip()

lm_pos = set(lm_df.loc[lm_df["Positive"] > 0, "Word"])
lm_neg = set(lm_df.loc[lm_df["Negative"] > 0, "Word"])
lm_unc = set(lm_df.loc[lm_df["Uncertainty"] > 0, "Word"])

print("LM positive words:", len(lm_pos))
print("LM negative words:", len(lm_neg))
print("LM uncertainty words:", len(lm_unc))

Harvard positive words: 1636
Harvard negative words: 2005
LM positive words: 347
LM negative words: 2345
LM uncertainty words: 297


# 6. Basic dictionary sentiment scoring

For each text:

*   split into words
*   count positive words
*   count negative words
* calculate tone = (Postive count - negative count) / total words










In [6]:
def score_text_basic(text, pos_words, neg_words, unc_words=None):
    words = text.split()

    pos_count = 0
    neg_count = 0
    unc_count = 0

    for word in words:
        if word in pos_words:
            pos_count += 1
        if word in neg_words:
            neg_count += 1
        if unc_words is not None and word in unc_words:
            unc_count += 1

    return pos_count, neg_count, unc_count

def tone_score(pos_count, neg_count, total_words):
    if total_words == 0:
        return np.nan
    return (pos_count - neg_count) / total_words

def add_dictionary_scores(df, text_col, word_count_col, section_name, pos_words, neg_words, prefix, unc_words=None):
    pos_list = []
    neg_list = []
    unc_list = []
    tone_list = []

    for i in range(len(df)):
        text = df.iloc[i][text_col]
        total_words = df.iloc[i][word_count_col]

        pos_count, neg_count, unc_count = score_text_basic(text, pos_words, neg_words, unc_words)
        tone = tone_score(pos_count, neg_count, total_words)

        pos_list.append(pos_count)
        neg_list.append(neg_count)
        unc_list.append(unc_count)
        tone_list.append(tone)

    df[f"{prefix}_{section_name}_pos"] = pos_list
    df[f"{prefix}_{section_name}_neg"] = neg_list
    df[f"{prefix}_{section_name}_tone"] = tone_list

    if unc_words is not None:
        df[f"{prefix}_{section_name}_unc"] = unc_list

# Harvard scores
add_dictionary_scores(master_df, "presentation_clean", "presentation_n_words", "presentation", harvard_pos, harvard_neg, "harvard")
add_dictionary_scores(master_df, "qa_answers_clean", "qa_answers_n_words", "qa_answers", harvard_pos, harvard_neg, "harvard")
add_dictionary_scores(master_df, "full_call_clean", "full_call_n_words", "full_call", harvard_pos, harvard_neg, "harvard")

# LM scores
add_dictionary_scores(master_df, "presentation_clean", "presentation_n_words", "presentation", lm_pos, lm_neg, "lm", lm_unc)
add_dictionary_scores(master_df, "qa_answers_clean", "qa_answers_n_words", "qa_answers", lm_pos, lm_neg, "lm", lm_unc)
add_dictionary_scores(master_df, "full_call_clean", "full_call_n_words", "full_call", lm_pos, lm_neg, "lm", lm_unc)

print(master_df[[
    "harvard_presentation_tone",
    "harvard_qa_answers_tone",
    "harvard_full_call_tone",
    "lm_presentation_tone",
    "lm_qa_answers_tone",
    "lm_full_call_tone"
]].describe())

       harvard_presentation_tone  harvard_qa_answers_tone  \
count                2876.000000              2877.000000   
mean                    0.032947                 0.030402   
std                     0.010364                 0.010019   
min                     0.000476                -0.011548   
25%                     0.025756                 0.024093   
50%                     0.032850                 0.030108   
75%                     0.039487                 0.036823   
max                     0.092105                 0.089552   

       harvard_full_call_tone  lm_presentation_tone  lm_qa_answers_tone  \
count             2877.000000           2876.000000         2877.000000   
mean                 0.032581              0.012201            0.008215   
std                  0.007477              0.008155            0.005966   
min                  0.005699             -0.037139           -0.015385   
25%                  0.027735              0.006929            0.004389   


# 7. Negation-adjusted sentiment


In [7]:
negation_words = {
    "not", "no", "never", "none", "neither", "nor", "without",
    "cannot", "can't", "couldn't", "didn't", "doesn't", "don't",
    "hadn't", "hasn't", "haven't", "isn't", "aren't", "wasn't",
    "weren't", "won't", "wouldn't", "shouldn't", "mustn't",
    "needn't", "hardly", "barely", "scarcely"
}

def is_negation(word):
    return word in negation_words or word.endswith("n't")

def score_text_with_negation(text, pos_words, neg_words, window=3):
    words = text.split()

    pos_count = 0
    neg_count = 0

    for i, word in enumerate(words):
        left_words = words[max(0, i - window):i]

        negated = False
        for left_word in left_words:
            if is_negation(left_word):
                negated = True
                break

        if word in pos_words:
            if negated:
                neg_count += 1
            else:
                pos_count += 1

        if word in neg_words:
            if negated:
                pos_count += 1
            else:
                neg_count += 1

    return pos_count, neg_count

def add_negation_scores(df, text_col, word_count_col, section_name, pos_words, neg_words, prefix, window=3):
    pos_list = []
    neg_list = []
    tone_list = []

    for i in range(len(df)):
        text = df.iloc[i][text_col]
        total_words = df.iloc[i][word_count_col]

        pos_count, neg_count = score_text_with_negation(text, pos_words, neg_words, window)
        tone = tone_score(pos_count, neg_count, total_words)

        pos_list.append(pos_count)
        neg_list.append(neg_count)
        tone_list.append(tone)

    df[f"{prefix}_negation_{section_name}_pos"] = pos_list
    df[f"{prefix}_negation_{section_name}_neg"] = neg_list
    df[f"{prefix}_negation_{section_name}_tone"] = tone_list

# Harvard negation scores
add_negation_scores(master_df, "presentation_clean", "presentation_n_words", "presentation", harvard_pos, harvard_neg, "harvard")
add_negation_scores(master_df, "qa_answers_clean", "qa_answers_n_words", "qa_answers", harvard_pos, harvard_neg, "harvard")
add_negation_scores(master_df, "full_call_clean", "full_call_n_words", "full_call", harvard_pos, harvard_neg, "harvard")

# LM negation scores
add_negation_scores(master_df, "presentation_clean", "presentation_n_words", "presentation", lm_pos, lm_neg, "lm")
add_negation_scores(master_df, "qa_answers_clean", "qa_answers_n_words", "qa_answers", lm_pos, lm_neg, "lm")
add_negation_scores(master_df, "full_call_clean", "full_call_n_words", "full_call", lm_pos, lm_neg, "lm")

print(master_df[[
    "harvard_negation_presentation_tone",
    "harvard_negation_qa_answers_tone",
    "harvard_negation_full_call_tone",
    "lm_negation_presentation_tone",
    "lm_negation_qa_answers_tone",
    "lm_negation_full_call_tone"
]].describe())

       harvard_negation_presentation_tone  harvard_negation_qa_answers_tone  \
count                         2876.000000                       2877.000000   
mean                             0.032483                          0.028424   
std                              0.010316                          0.009919   
min                              0.001428                         -0.012831   
25%                              0.025286                          0.022109   
50%                              0.032373                          0.028174   
75%                              0.039177                          0.034550   
max                              0.092105                          0.074627   

       harvard_negation_full_call_tone  lm_negation_presentation_tone  \
count                      2877.000000                    2876.000000   
mean                          0.031238                       0.012264   
std                           0.007468                       0.008114

# 8. Build a simple custom dictionary


*   finding common words in the earnings-call texts
*   selecting words that start with positive
*   stems like grow, improv, strong
*   selecting words that start with negative
*   stems like declin, risk, weak
*   adding a few manual words
*   removing bad matches like software




In [8]:

# Count how often each word appears in the full-call corpus
all_words = []

for text in master_df["full_call_clean"]:
    words = text.split()
    all_words.extend(words)

word_freq = pd.Series(all_words).value_counts()

# Keep words that appear at least 10 times
common_words = set(word_freq[word_freq >= 10].index)

# Starter stems
positive_stems = [
    "grow", "improv", "strong", "confiden", "optim", "resilien",
    "momentum", "exceed", "efficien", "robust", "accelerat",
    "record", "benefit", "opportun", "healthy", "solid",
    "expand", "success", "favorable", "outperform", "strength",
    "better", "best", "recover", "profitab", "upside", "tailwind"
]

negative_stems = [
    "declin", "weak", "pressur", "headwind", "challeng", "uncertain",
    "slow", "risk", "inflation", "disrupt", "lower", "soft",
    "constrain", "impair", "volatile", "miss", "downturn",
    "adverse", "difficult", "wors", "shortfall", "cautio", "delay"
]

# Manual words to include
manual_pos = {
    "synergy", "synergies", "disciplined", "discipline",
    "stabilized", "stabilizing", "cashflow"
}

manual_neg = {
    "downside", "downsides", "marginpressure",
    "uncertainty", "slowdown", "slowdowns",
    "weakness", "challenging", "constraints", "constrained",
    "deterioration", "deteriorate"
}

def words_from_stems(vocab_set, stem_list):
    selected_words = set()

    for word in vocab_set:
        for stem in stem_list:
            if word.startswith(stem):
                selected_words.add(word)
                break

    return selected_words

custom_pos = words_from_stems(common_words, positive_stems)
custom_neg = words_from_stems(common_words, negative_stems)

custom_pos = custom_pos | manual_pos
custom_neg = custom_neg | manual_neg

# Keep only words that are really in the corpus
custom_pos = {word for word in custom_pos if word in common_words}
custom_neg = {word for word in custom_neg if word in common_words}

# Remove obvious false matches
custom_neg = custom_neg - {"software", "softwares", "mission", "missions"}

# Remove overlap
overlap_words = custom_pos & custom_neg
custom_pos = custom_pos - overlap_words
custom_neg = custom_neg - overlap_words

print("Custom positive words:", len(custom_pos))
print("Custom negative words:", len(custom_neg))
print("Overlap removed:", len(overlap_words))

print("\nTop custom positive words:")
print(word_freq[word_freq.index.isin(custom_pos)].head(20))

print("\nTop custom negative words:")
print(word_freq[word_freq.index.isin(custom_neg)].head(20))

Custom positive words: 131
Custom negative words: 119
Overlap removed: 0

Top custom positive words:
growth           92317
strong           37955
better           20298
opportunity      14978
opportunities    13956
benefit          11873
improvement      11400
grow             11342
growing          10665
best              9806
improve           8139
strength          7531
improved          7191
solid             6828
momentum          6731
benefits          6696
record            6414
confident         5578
success           5244
improving         4892
Name: count, dtype: int64

Top custom negative words:
lower            20464
decline           8228
risk              7359
pressure          4493
declined          4229
headwind          3735
difficult         3652
challenges        3612
headwinds         3473
inflation         3157
risks             3005
challenging       2793
declines          2752
uncertainties     2355
uncertainty       2184
challenge         1520
slow             

#9. Score the custom dictionary

In [9]:
add_dictionary_scores(master_df, "presentation_clean", "presentation_n_words", "presentation", custom_pos, custom_neg, "custom")
add_dictionary_scores(master_df, "qa_answers_clean", "qa_answers_n_words", "qa_answers", custom_pos, custom_neg, "custom")
add_dictionary_scores(master_df, "full_call_clean", "full_call_n_words", "full_call", custom_pos, custom_neg, "custom")

print(master_df[[
    "custom_presentation_tone",
    "custom_qa_answers_tone",
    "custom_full_call_tone"
]].describe())


       custom_presentation_tone  custom_qa_answers_tone  custom_full_call_tone
count               2876.000000             2877.000000            2877.000000
mean                   0.015507                0.010120               0.011668
std                    0.008626                0.005137               0.005446
min                   -0.023384               -0.004175              -0.004843
25%                    0.009715                0.006590               0.007956
50%                    0.015244                0.009858               0.011471
75%                    0.021100                0.013505               0.015131
max                    0.050432                0.032389               0.035609


# 10. Check correlations

Looks at how similar the sentiment measures are to each other.

In [10]:
corr_cols = [
    "harvard_presentation_tone",
    "harvard_qa_answers_tone",
    "harvard_full_call_tone",
    "harvard_negation_presentation_tone",
    "harvard_negation_qa_answers_tone",
    "harvard_negation_full_call_tone",
    "lm_presentation_tone",
    "lm_qa_answers_tone",
    "lm_full_call_tone",
    "lm_negation_presentation_tone",
    "lm_negation_qa_answers_tone",
    "lm_negation_full_call_tone",
    "custom_presentation_tone",
    "custom_qa_answers_tone",
    "custom_full_call_tone"
]

print(master_df[corr_cols].corr())

                                    harvard_presentation_tone  \
harvard_presentation_tone                            1.000000   
harvard_qa_answers_tone                              0.363879   
harvard_full_call_tone                               0.723084   
harvard_negation_presentation_tone                   0.995399   
harvard_negation_qa_answers_tone                     0.366365   
harvard_negation_full_call_tone                      0.716315   
lm_presentation_tone                                 0.391402   
lm_qa_answers_tone                                   0.213033   
lm_full_call_tone                                    0.352375   
lm_negation_presentation_tone                        0.392190   
lm_negation_qa_answers_tone                          0.209197   
lm_negation_full_call_tone                           0.352383   
custom_presentation_tone                             0.202848   
custom_qa_answers_tone                               0.126087   
custom_full_call_tone    

# 11. Prepare regression data


*  creates log_atq as a size control
*  lists sentiment variable
*  standardizes them into z-scores
*  defines market-reaction outcomes


In [11]:
reg_df = master_df.copy()

# Size control
reg_df["log_atq"] = np.where(reg_df["atq"] > 0, np.log(reg_df["atq"]), np.nan)

sentiment_vars = [
    "harvard_presentation_tone",
    "harvard_qa_answers_tone",
    "harvard_full_call_tone",
    "harvard_negation_presentation_tone",
    "harvard_negation_qa_answers_tone",
    "harvard_negation_full_call_tone",
    "lm_presentation_tone",
    "lm_qa_answers_tone",
    "lm_full_call_tone",
    "lm_negation_presentation_tone",
    "lm_negation_qa_answers_tone",
    "lm_negation_full_call_tone",
    "custom_presentation_tone",
    "custom_qa_answers_tone",
    "custom_full_call_tone"
]

# Standardize sentiment variables so coefficients are easier to compare
for col in sentiment_vars:
    mean_value = reg_df[col].mean()
    std_value = reg_df[col].std()

    if pd.notna(std_value) and std_value > 0:
        reg_df[col + "_z"] = (reg_df[col] - mean_value) / std_value
    else:
        reg_df[col + "_z"] = np.nan

controls = ["surp", "hvol", "IV_l1d", "log_atq"]

outcomes = [
    "CAR01-ff3",
    "CAR01-Carhart",
    "CAR-11-ff3",
    "CAR-11-Carhart"
]

print("Missing values in outcomes and controls:")
print(reg_df[outcomes + controls].isna().mean().sort_values())

Missing values in outcomes and controls:
log_atq           0.000000
surp              0.000000
CAR01-ff3         0.020855
CAR01-Carhart     0.020855
CAR-11-Carhart    0.020855
CAR-11-ff3        0.020855
hvol              0.035106
IV_l1d            0.035106
dtype: float64


# 12. Run OLS regressions

*   coefficient
*   t-statistic
*   p-value
*   adjusted R²
*   control variables










In [12]:
def run_ols(df, y_col, x_col, control_cols):
    use_cols = [y_col, x_col] + control_cols
    temp_df = df[use_cols].copy()

    # Make sure all columns are numeric
    for col in use_cols:
        temp_df[col] = pd.to_numeric(temp_df[col], errors="coerce")

    temp_df = temp_df.dropna()

    # Skip if too few rows
    if len(temp_df) < 50:
        return {
            "outcome": y_col,
            "x": x_col,
            "n": len(temp_df),
            "coef": np.nan,
            "t": np.nan,
            "p": np.nan,
            "r2": np.nan,
            "adj_r2": np.nan
        }

    y = temp_df[y_col]
    X = temp_df[[x_col] + control_cols]
    X = sm.add_constant(X)

    # HC3 robust standard errors
    model = sm.OLS(y, X).fit(cov_type="HC3")

    return {
        "outcome": y_col,
        "x": x_col,
        "n": int(model.nobs),
        "coef": model.params[x_col],
        "t": model.tvalues[x_col],
        "p": model.pvalues[x_col],
        "r2": model.rsquared,
        "adj_r2": model.rsquared_adj
    }

def get_family(name):
    name = name.replace("_z", "")

    if name.startswith("harvard_negation_"):
        return "Harvard + negation"
    elif name.startswith("harvard_"):
        return "Harvard"
    elif name.startswith("lm_negation_"):
        return "LM + negation"
    elif name.startswith("lm_"):
        return "LM"
    elif name.startswith("custom_"):
        return "Custom"
    else:
        return "Other"

def get_section(name):
    name = name.replace("_z", "")

    if "presentation" in name:
        return "Presentation"
    elif "qa_answers" in name:
        return "Q&A answers"
    elif "full_call" in name:
        return "Full call"
    else:
        return "Other"

all_results = []

for y_col in outcomes:
    for sent_col in sentiment_vars:
        result = run_ols(reg_df, y_col, sent_col + "_z", controls)
        all_results.append(result)

results_df = pd.DataFrame(all_results)
results_df["family"] = results_df["x"].apply(get_family)
results_df["section"] = results_df["x"].apply(get_section)
results_df["abs_t"] = results_df["t"].abs()

for y_col in outcomes:
    print("\nOutcome:", y_col)
    best_models = results_df[results_df["outcome"] == y_col].sort_values(["p", "abs_t"])
    print(best_models[["family", "section", "x", "coef", "t", "p", "adj_r2", "n"]].head(10))



Outcome: CAR01-ff3
                family       section                                  x  \
14              Custom     Full call            custom_full_call_tone_z   
11       LM + negation     Full call       lm_negation_full_call_tone_z   
8                   LM     Full call                lm_full_call_tone_z   
12              Custom  Presentation         custom_presentation_tone_z   
9        LM + negation  Presentation    lm_negation_presentation_tone_z   
6                   LM  Presentation             lm_presentation_tone_z   
7                   LM   Q&A answers               lm_qa_answers_tone_z   
10       LM + negation   Q&A answers      lm_negation_qa_answers_tone_z   
13              Custom   Q&A answers           custom_qa_answers_tone_z   
5   Harvard + negation     Full call  harvard_negation_full_call_tone_z   

        coef         t             p    adj_r2     n  
14  0.007170  6.657358  2.787939e-11  0.046483  2773  
11  0.007315  6.250761  4.084570e-10  0.0471

# PART 2

# 1) Install and import

In [13]:
# Install Hugging Face libraries in Colab
!pip install -q transformers accelerate sentencepiece

In [14]:
import pandas as pd
import numpy as np
import re
import torch
import statsmodels.api as sm
from transformers import pipeline

# 2) Dataset

In [15]:
# Use a separate dataframe for FinBERT
finbert_df = master_df.copy()

# Keep the same three text versions used in Part 1
text_cols = ["presentation_text", "qa_answers_text", "full_call_text"]

for col in text_cols:
    finbert_df[col] = finbert_df[col].fillna("").astype(str)

# 3) Light cleaning for FinBERT

In [16]:
def light_clean_text(text):
    text = str(text)
    text = re.sub(r"[\r\n]+", " ", text)   # remove line breaks
    text = re.sub(r"\s+", " ", text)       # remove extra spaces
    return text.strip()

for col in text_cols:
    finbert_df[col] = finbert_df[col].apply(light_clean_text)

# 4) Load FinBERT from Hugging Face

In [17]:
# Use GPU in Colab if available
device_num = 0 if torch.cuda.is_available() else -1

# Load FinBERT directly from Hugging Face Hub
finbert_pipe = pipeline(
    task="text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    device=device_num
)

print("FinBERT loaded")
print("Using GPU:", torch.cuda.is_available())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

FinBERT loaded
Using GPU: True


# 5) Split long transcripts into chunks

In [18]:
def split_text_into_chunks(text, words_per_chunk=220):
    words = text.split()

    if len(words) == 0:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = start + words_per_chunk
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end

    return chunks

# 6) Get positive, negative, and neutral probabilities

In [19]:
def get_label_scores(result_list):
    pos_score = 0.0
    neg_score = 0.0
    neu_score = 0.0

    for item in result_list:
        label = item["label"].lower()
        score = item["score"]

        if label == "positive":
            pos_score = score
        elif label == "negative":
            neg_score = score
        elif label == "neutral":
            neu_score = score

    return pos_score, neg_score, neu_score

# 7) Score one transcript with FinBERT

FinBERT tone=positive probability−negative probability

In [20]:
def score_text_with_finbert(text, batch_size=16):
    chunks = split_text_into_chunks(text, words_per_chunk=220)

    if len(chunks) == 0:
        return {
            "finbert_pos": np.nan,
            "finbert_neg": np.nan,
            "finbert_neu": np.nan,
            "finbert_tone": np.nan,
            "n_chunks": 0
        }

    # Count words in each chunk for weighted average
    chunk_word_counts = []
    for chunk in chunks:
        chunk_word_counts.append(len(chunk.split()))

    # Run FinBERT on all chunks
    results = finbert_pipe(
        chunks,
        top_k=None,          # return all labels
        truncation=True,
        max_length=512,
        batch_size=batch_size
    )

    pos_list = []
    neg_list = []
    neu_list = []

    for one_result in results:
        pos_score, neg_score, neu_score = get_label_scores(one_result)
        pos_list.append(pos_score)
        neg_list.append(neg_score)
        neu_list.append(neu_score)

    total_words = sum(chunk_word_counts)

    weighted_pos = 0.0
    weighted_neg = 0.0
    weighted_neu = 0.0

    for i in range(len(chunks)):
        weight = chunk_word_counts[i] / total_words
        weighted_pos += pos_list[i] * weight
        weighted_neg += neg_list[i] * weight
        weighted_neu += neu_list[i] * weight

    finbert_tone = weighted_pos - weighted_neg

    return {
        "finbert_pos": weighted_pos,
        "finbert_neg": weighted_neg,
        "finbert_neu": weighted_neu,
        "finbert_tone": finbert_tone,
        "n_chunks": len(chunks)
    }

# 8) Apply FinBERT to presentation, Q&A answers, and full call

In [21]:
def add_finbert_scores(df, text_col, prefix):
    pos_values = []
    neg_values = []
    neu_values = []
    tone_values = []
    chunk_values = []

    for i in range(len(df)):
        text = df.iloc[i][text_col]
        scores = score_text_with_finbert(text)

        pos_values.append(scores["finbert_pos"])
        neg_values.append(scores["finbert_neg"])
        neu_values.append(scores["finbert_neu"])
        tone_values.append(scores["finbert_tone"])
        chunk_values.append(scores["n_chunks"])

        if (i + 1) % 50 == 0:
            print("Finished", i + 1, "rows for", prefix)

    df[f"{prefix}_finbert_pos"] = pos_values
    df[f"{prefix}_finbert_neg"] = neg_values
    df[f"{prefix}_finbert_neu"] = neu_values
    df[f"{prefix}_finbert_tone"] = tone_values
    df[f"{prefix}_finbert_n_chunks"] = chunk_values

In [22]:
# Run FinBERT for the three corpus versions
add_finbert_scores(finbert_df, "presentation_text", "presentation")
add_finbert_scores(finbert_df, "qa_answers_text", "qa_answers")
add_finbert_scores(finbert_df, "full_call_text", "full_call")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Finished 50 rows for presentation
Finished 100 rows for presentation
Finished 150 rows for presentation
Finished 200 rows for presentation
Finished 250 rows for presentation
Finished 300 rows for presentation
Finished 350 rows for presentation
Finished 400 rows for presentation
Finished 450 rows for presentation
Finished 500 rows for presentation
Finished 550 rows for presentation
Finished 600 rows for presentation
Finished 650 rows for presentation
Finished 700 rows for presentation
Finished 750 rows for presentation
Finished 800 rows for presentation
Finished 850 rows for presentation
Finished 900 rows for presentation
Finished 950 rows for presentation
Finished 1000 rows for presentation
Finished 1050 rows for presentation
Finished 1100 rows for presentation
Finished 1150 rows for presentation
Finished 1200 rows for presentation
Finished 1250 rows for presentation
Finished 1300 rows for presentation
Finished 1350 rows for presentation
Finished 1400 rows for presentation
Finished 145

In [23]:
# Quick summary
print(finbert_df[[
    "presentation_finbert_tone",
    "qa_answers_finbert_tone",
    "full_call_finbert_tone"
]].describe())

       presentation_finbert_tone  qa_answers_finbert_tone  \
count                2876.000000              2877.000000   
mean                    0.392603                 0.251570   
std                     0.264879                 0.150787   
min                    -0.703352                -0.452617   
25%                     0.224281                 0.151766   
50%                     0.437583                 0.250802   
75%                     0.592627                 0.349449   
max                     0.916861                 0.747513   

       full_call_finbert_tone  
count             2877.000000  
mean                 0.268022  
std                  0.152739  
min                 -0.386842  
25%                  0.172177  
50%                  0.277840  
75%                  0.376309  
max                  0.719369  


# 9) Standardize the FinBERT scores

In [24]:
finbert_vars = [
    "presentation_finbert_tone",
    "qa_answers_finbert_tone",
    "full_call_finbert_tone"
]

for col in finbert_vars:
    mean_value = finbert_df[col].mean()
    std_value = finbert_df[col].std()

    if pd.notna(std_value) and std_value > 0:
        finbert_df[col + "_z"] = (finbert_df[col] - mean_value) / std_value
    else:
        finbert_df[col + "_z"] = np.nan

# 10) Run the regressions

In [25]:
controls = ["surp", "hvol", "IV_l1d", "log_atq"]

outcomes = [
    "CAR01-ff3",
    "CAR01-Carhart",
    "CAR-11-ff3",
    "CAR-11-Carhart"
]

In [26]:
def run_ols(df, y_col, x_col, control_cols):
    use_cols = [y_col, x_col] + control_cols
    temp_df = df[use_cols].copy()

    for col in use_cols:
        temp_df[col] = pd.to_numeric(temp_df[col], errors="coerce")

    temp_df = temp_df.dropna()

    if len(temp_df) < 50:
        return {
            "outcome": y_col,
            "x": x_col,
            "n": len(temp_df),
            "coef": np.nan,
            "t": np.nan,
            "p": np.nan,
            "adj_r2": np.nan
        }

    y = temp_df[y_col]
    X = temp_df[[x_col] + control_cols]
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit(cov_type="HC3")

    return {
        "outcome": y_col,
        "x": x_col,
        "n": int(model.nobs),
        "coef": model.params[x_col],
        "t": model.tvalues[x_col],
        "p": model.pvalues[x_col],
        "adj_r2": model.rsquared_adj
    }

In [28]:
# Create log_atq if atq exists
if "atq" in finbert_df.columns:
    finbert_df["log_atq"] = np.where(finbert_df["atq"] > 0, np.log(finbert_df["atq"]), np.nan)
else:
    print("Column 'atq' is missing, so log_atq cannot be created.")

In [29]:
finbert_results = []

for y_col in outcomes:
    for x_col in [
        "presentation_finbert_tone_z",
        "qa_answers_finbert_tone_z",
        "full_call_finbert_tone_z"
    ]:
        result = run_ols(finbert_df, y_col, x_col, controls)
        finbert_results.append(result)

finbert_results_df = pd.DataFrame(finbert_results)
print(finbert_results_df.sort_values(["outcome", "p"]))

           outcome                            x     n      coef         t  \
11  CAR-11-Carhart     full_call_finbert_tone_z  2773  0.007070  6.385915   
9   CAR-11-Carhart  presentation_finbert_tone_z  2772  0.006683  5.918233   
10  CAR-11-Carhart    qa_answers_finbert_tone_z  2773  0.004124  3.556584   
8       CAR-11-ff3     full_call_finbert_tone_z  2773  0.007149  6.419981   
6       CAR-11-ff3  presentation_finbert_tone_z  2772  0.006707  5.886121   
7       CAR-11-ff3    qa_answers_finbert_tone_z  2773  0.004271  3.650292   
5    CAR01-Carhart     full_call_finbert_tone_z  2773  0.007149  6.419981   
3    CAR01-Carhart  presentation_finbert_tone_z  2772  0.006707  5.886121   
4    CAR01-Carhart    qa_answers_finbert_tone_z  2773  0.004271  3.650292   
2        CAR01-ff3     full_call_finbert_tone_z  2773  0.007409  6.776460   
0        CAR01-ff3  presentation_finbert_tone_z  2772  0.006990  6.400614   
1        CAR01-ff3    qa_answers_finbert_tone_z  2773  0.004489  3.943417   

# 11) Compare FinBERT with your best Bag-of-Words measures


In [30]:
compare_vars = [
    "lm_full_call_tone_z",
    "lm_negation_full_call_tone_z",
    "custom_full_call_tone_z",
    "full_call_finbert_tone_z"
]

available_compare_vars = []

for col in compare_vars:
    if col in finbert_df.columns:
        available_compare_vars.append(col)

comparison_results = []

for y_col in outcomes:
    for x_col in available_compare_vars:
        result = run_ols(finbert_df, y_col, x_col, controls)
        comparison_results.append(result)

comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.sort_values(["outcome", "p"]))

          outcome                         x     n      coef         t  \
3  CAR-11-Carhart  full_call_finbert_tone_z  2773  0.007070  6.385915   
2      CAR-11-ff3  full_call_finbert_tone_z  2773  0.007149  6.419981   
1   CAR01-Carhart  full_call_finbert_tone_z  2773  0.007149  6.419981   
0       CAR01-ff3  full_call_finbert_tone_z  2773  0.007409  6.776460   

              p    adj_r2  
3  1.703754e-10  0.045210  
2  1.362913e-10  0.043768  
1  1.362913e-10  0.043768  
0  1.231563e-11  0.047481  
